In [1]:
import pandas as pd, numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import TargetEncoder
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

df = pd.read_parquet("../out/ksa_sales_clean.parquet")
m = df[(df.classification == "سكني") & (~df.is_multi_property)
       & (df.area_m2.between(50, 5_000))
       & (df.price_sar.between(50_000, 20_000_000))].copy()

# features
m["log_area"] = np.log(m.area_m2)
m["t"] = (m.date_greg - pd.Timestamp("2020-01-01")).dt.days / 365.25
y = np.log1p(m.price_sar)

cut = pd.Timestamp("2025-01-01")
itr, ite = m.date_greg < cut, m.date_greg >= cut
X = m[["log_area", "t", "location", "city", "region"]]

pre = ColumnTransformer([
    ("num", "passthrough", ["log_area", "t"]),
    ("cat", TargetEncoder(target_type="continuous", random_state=0),
            ["location", "city", "region"]),
])

pipe = Pipeline([
    ("pre", pre),
    ("model", HistGradientBoostingRegressor(
        max_iter=400, learning_rate=0.1, min_samples_leaf=50, random_state=0)),
])

pipe.fit(X[itr], y[itr])
p = pipe.predict(X[ite])

pred, actual = np.expm1(p), m.price_sar[ite]
print("MODEL, test = 2025+")
print("  R²  (log):", round(r2_score(y[ite], p), 4))
print("  MAE (log):", round(mean_absolute_error(y[ite], p), 4))
print("  median abs % error:", f"{np.median(np.abs(pred-actual)/actual):.1%}")
print("  bias:", round(float(np.median(pred/actual)), 3))

MODEL, test = 2025+
  R²  (log): 0.661
  MAE (log): 0.4124
  median abs % error: 27.8%
  bias: 0.989


In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

def run(name, model, scale):
    pre = ColumnTransformer([
        ("num", StandardScaler() if scale else "passthrough", ["log_area", "t"]),
        ("cat", TargetEncoder(target_type="continuous"), ["location", "city", "region"]),
    ])
    pipe = Pipeline([("pre", pre), ("model", model)])
    pipe.fit(X[itr], y[itr])
    p = pipe.predict(X[ite])
    pred, actual = np.expm1(p), m.price_sar[ite]
    print(f"{name:22s} R² {r2_score(y[ite], p):.4f} | MAE {mean_absolute_error(y[ite], p):.4f} "
          f"| med%err {np.median(np.abs(pred-actual)/actual):.1%} "
          f"| bias {np.median(pred/actual):.3f}")

run("Ridge (linear)", Ridge(alpha=1.0), True)
run("HistGradientBoosting", HistGradientBoostingRegressor(
        max_iter=400, learning_rate=0.1, min_samples_leaf=50, random_state=0), False)

Ridge (linear)         R² 0.5953 | MAE 0.4689 | med%err 34.5% | bias 1.076
HistGradientBoosting   R² 0.6611 | MAE 0.4119 | med%err 27.6% | bias 0.988
